# Implied Volatility Surface Forecasting - Colab Training Notebook

This notebook is fully self-contained and ready to run on Google Colab (with a **T4 GPU runtime** enabled). It will:
1. Install necessary dependencies (`arch`, `mlflow`, `yfinance`, etc.).
2. Define the deep learning architectures (`ConvLSTM`, `LSTM`, `Transformer`).
3. Define the custom `SmoothnessRegularizedLoss` incorporating strike/expiry spatial penalties.
4. Implement all econometric benchmarks (`Random Walk`, `Historical Mean`, `Exponential Smoothing`, `GARCH`, `HAR-RV`).
5. Evolve synthetic surfaces, run the comparative training loops, and output the final results table.
6. Save the trained champion weights as `checkpoint.pt` for FastAPI serving.

### Enable GPU in Colab:
Go to **Runtime** -> **Change runtime type** -> select **T4 GPU**.

In [12]:
# 1. Install required libraries
!pip install arch mlflow yfinance pyyaml scipy scikit-learn pandas numpy torch

In [13]:
!nvidia-smi
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))
print("Current Device:", torch.cuda.current_device())

Fri May 29 05:30:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             31W /   70W |     269MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# 2. Imports
import os
import math
import yaml
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu128
CUDA Available: True


## Define Neural Networks

In [15]:
# 3. Deep Learning Architectures

class StackedLSTM(nn.Module):
    def __init__(self, grid_size=(7, 7), hidden_dim=256, num_layers=3, horizon=1):
        super().__init__()
        self.M, self.N = grid_size
        self.input_dim = self.M * self.N
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.horizon = horizon
        self.lstm = nn.LSTM(input_size=self.input_dim, hidden_size=self.hidden_dim, num_layers=self.num_layers, batch_first=True)
        self.decoder = nn.Sequential(
            nn.Linear(self.hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, self.horizon * self.input_dim)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        x_flat = x.view(B, L, -1)
        out, _ = self.lstm(x_flat)
        last_hidden = out[:, -1, :]
        decoded = self.decoder(last_hidden)
        return decoded.view(B, self.horizon, self.M, self.N)


class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_dim, kernel_size=3, bias=True):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels=self.in_channels + self.hidden_dim,
            out_channels=4 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=bias
        )
    def forward(self, x, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([x, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next
    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device)
        )


class ConvLSTM(nn.Module):
    def __init__(self, in_channels=1, hidden_dims=[32, 64, 64], kernel_size=3, num_layers=3, horizon=1):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_dims = hidden_dims
        self.num_layers = num_layers
        self.horizon = horizon
        cell_list = []
        for i in range(self.num_layers):
            cur_in = self.in_channels if i == 0 else self.hidden_dims[i - 1]
            cell_list.append(ConvLSTMCell(in_channels=cur_in, hidden_dim=self.hidden_dims[i], kernel_size=kernel_size))
        self.cell_list = nn.ModuleList(cell_list)
        bn_list = []
        for i in range(self.num_layers - 1):
            bn_list.append(nn.BatchNorm3d(self.hidden_dims[i]))
        self.bn_list = nn.ModuleList(bn_list)
        self.decoder = nn.Sequential(
            nn.Conv2d(self.hidden_dims[-1], 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, self.horizon, kernel_size=1, padding=0)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        device = x.device
        current_input = x
        for i in range(self.num_layers):
            cell = self.cell_list[i]
            h, c = cell.init_hidden(B, (M, N), device)
            outputs = []
            for t in range(L):
                h, c = cell(current_input[:, t, :, :, :], (h, c))
                outputs.append(h)
            outputs = torch.stack(outputs, dim=1)
            if i < self.num_layers - 1:
                outputs = outputs.permute(0, 2, 1, 3, 4)
                outputs = self.bn_list[i](outputs)
                outputs = outputs.permute(0, 2, 1, 3, 4)
            current_input = outputs
        last_hidden = current_input[:, -1, :, :, :]
        return self.decoder(last_hidden)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerEncoderModel(nn.Module):
    def __init__(self, grid_size=(7, 7), d_model=128, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1, horizon=1):
        super().__init__()
        self.M, self.N = grid_size
        self.input_dim = self.M * self.N
        self.d_model = d_model
        self.horizon = horizon
        self.embedding = nn.Linear(self.input_dim, self.d_model)
        self.pos_encoder = PositionalEncoding(d_model=self.d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=self.d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Sequential(
            nn.Linear(self.d_model, 256),
            nn.ReLU(),
            nn.Linear(256, self.horizon * self.input_dim)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        x_flat = x.view(B, L, -1)
        embedded = self.embedding(x_flat)
        encoded = self.pos_encoder(embedded)
        out = self.transformer_encoder(encoded)
        last_step = out[:, -1, :]
        decoded = self.decoder(last_step)
        return decoded.view(B, self.horizon, self.M, self.N)

## Define Custom Loss Function

In [16]:
# 4. Smoothness-Regularized Loss Function

class SmoothnessRegularizedLoss(nn.Module):
    def __init__(self, lambda_strike=0.01, lambda_expiry=0.01, lambda_calendar=0.0, lambda_butterfly=0.0, grid_taus=None):
        super().__init__()
        self.lambda_strike = lambda_strike
        self.lambda_expiry = lambda_expiry
        self.lambda_calendar = lambda_calendar
        self.lambda_butterfly = lambda_butterfly
        
        taus = grid_taus if grid_taus is not None else np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0])
        self.register_buffer("grid_taus_tensor", torch.from_numpy(taus).float())

    def forward(self, y_pred, y_true):
        mse_loss = F.mse_loss(y_pred, y_true)

        # Strike smoothness (reflection padding on dim -2)
        padded_strike = F.pad(y_pred, pad=(0, 0, 1, 1), mode="reflect")
        strike_d2 = padded_strike[:, :, 2:, :] - 2 * padded_strike[:, :, 1:-1, :] + padded_strike[:, :, :-2, :]
        strike_loss = torch.mean(strike_d2 ** 2)

        # Expiry smoothness (reflection padding on dim -1)
        padded_expiry = F.pad(y_pred, pad=(1, 1, 0, 0), mode="reflect")
        expiry_d2 = padded_expiry[:, :, :, 2:] - 2 * padded_expiry[:, :, :, 1:-1] + padded_expiry[:, :, :, :-2]
        expiry_loss = torch.mean(expiry_d2 ** 2)

        # Arbitrage penalties
        calendar_loss = torch.tensor(0.0, device=y_pred.device)
        if self.lambda_calendar > 0:
            taus = self.grid_taus_tensor.view(1, 1, 1, -1)
            w = (y_pred ** 2) * taus
            w_diff = w[:, :, :, :-1] - w[:, :, :, 1:]
            calendar_loss = torch.mean(torch.relu(w_diff) ** 2)

        butterfly_loss = torch.tensor(0.0, device=y_pred.device)
        if self.lambda_butterfly > 0:
            d2 = y_pred[:, :, 2:, :] - 2 * y_pred[:, :, 1:-1, :] + y_pred[:, :, :-2, :]
            butterfly_loss = torch.mean(torch.relu(-d2) ** 2)

        total_loss = mse_loss + self.lambda_strike * strike_loss + self.lambda_expiry * expiry_loss + self.lambda_calendar * calendar_loss + self.lambda_butterfly * butterfly_loss
        return total_loss, {
            "mse_loss": mse_loss.item(),
            "strike_loss": strike_loss.item(),
            "expiry_loss": expiry_loss.item(),
            "total_loss": total_loss.item()
        }

## Define Baselines

In [17]:
# 5. Statistical & Econometric Baselines

class NaiveRandomWalk:
    def __init__(self, horizon=1):
        self.horizon = horizon
    def predict(self, X):
        last_step = X[:, -1, 0, :, :]
        return np.repeat(last_step[:, np.newaxis, :, :], self.horizon, axis=1)


class HistoricalMean:
    def __init__(self, horizon=1):
        self.horizon = horizon
    def predict(self, X):
        mean_surface = np.mean(X[:, :, 0, :, :], axis=1)
        return np.repeat(mean_surface[:, np.newaxis, :, :], self.horizon, axis=1)


class ExponentialSmoothing:
    def __init__(self, horizon=1, alpha=0.3):
        self.horizon = horizon
        self.alpha = alpha
    def predict(self, X):
        B, L, _, M, N = X.shape
        preds = np.zeros((B, self.horizon, M, N))
        for b in range(B):
            for i in range(M):
                for j in range(N):
                    ts = X[b, :, 0, i, j]
                    s = ts[0]
                    for t in range(1, L):
                        s = self.alpha * ts[t] + (1 - self.alpha) * s
                    preds[b, :, i, j] = s
        return preds


class GARCHModel:
    def __init__(self, horizon=1):
        self.horizon = horizon
    def predict(self, X):
        B, L, _, M, N = X.shape
        preds = np.zeros((B, self.horizon, M, N))
        try:
            from arch import arch_model
            for b in range(B):
                for i in range(M):
                    for j in range(N):
                        ts = X[b, :, 0, i, j]
                        try:
                            scaled = ts * 100.0
                            model = arch_model(scaled, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
                            res = model.fit(disp='off', show_warning=False)
                            forecasts = res.forecast(horizon=self.horizon, reindex=False)
                            var = forecasts.variance.values[0]
                            preds[b, :, i, j] = np.sqrt(np.clip(var, 1e-6, None)) / 100.0
                        except:
                            preds[b, :, i, j] = np.mean(ts)
        except ImportError:
            # Fallback to Historical Mean
            preds = np.mean(X[:, :, 0, :, :], axis=1)
            preds = np.repeat(preds[:, np.newaxis, :, :], self.horizon, axis=1)
        return preds


class HARRVModel:
    def __init__(self, horizon=1):
        self.horizon = horizon
        self.regressors = {}
    def fit(self, X, y):
        N_samples, L, _, M, N = X.shape
        for i in range(M):
            for j in range(N):
                features = []
                targets = []
                for s in range(N_samples):
                    ts = X[s, :, 0, i, j]
                    features.append([ts[-1], np.mean(ts[-5:]), np.mean(ts[-20:])])
                    targets.append(y[s, :, i, j])
                reg = LinearRegression()
                reg.fit(np.array(features), np.array(targets))
                self.regressors[(i, j)] = reg
    def predict(self, X):
        B, L, _, M, N = X.shape
        preds = np.zeros((B, self.horizon, M, N))
        for i in range(M):
            for j in range(N):
                features = []
                for b in range(B):
                    ts = X[b, :, 0, i, j]
                    features.append([ts[-1], np.mean(ts[-5:]), np.mean(ts[-20:])])
                reg = self.regressors.get((i, j))
                if reg is not None:
                    pred = reg.predict(np.array(features))
                    if self.horizon == 1:
                        pred = pred[:, np.newaxis]
                    preds[:, :, i, j] = pred
                else:
                    preds[:, :, i, j] = np.repeat(X[:, -1, 0, i, j][:, np.newaxis], self.horizon, axis=1)
        return np.clip(preds, 0.01, 5.0)

## Synthetic Data Generation & Sequencing

In [18]:
# 6. Data Generator & Sequence builders

GRID_KAPPAS = np.array([-0.30, -0.20, -0.10, 0.00, 0.10, 0.20, 0.30])
GRID_TAUS = np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0])

def generate_synthetic_dataset(num_days=300):
    level, skew, term = 0.22, 0.08, 0.04
    phi_l, phi_s, phi_t = 0.95, 0.90, 0.93
    mu_l, mu_s, mu_t = 0.20, 0.06, 0.03
    sigma_l, sigma_s, sigma_t = 0.015, 0.006, 0.004
    
    surfaces = []
    for day in range(num_days):
        level = mu_l + phi_l * (level - mu_l) + np.random.normal(0, sigma_l)
        skew = mu_s + phi_s * (skew - mu_s) + np.random.normal(0, sigma_s)
        term = mu_t + phi_t * (term - mu_t) + np.random.normal(0, sigma_t)
        
        level = np.clip(level, 0.08, 0.60)
        skew = np.clip(skew, 0.01, 0.20)
        term = np.clip(term, -0.05, 0.12)
        
        grid_iv = np.zeros((len(GRID_TAUS), len(GRID_KAPPAS)))
        for i, tau in enumerate(GRID_TAUS):
            for j, kappa in enumerate(GRID_KAPPAS):
                iv = level - skew * kappa + 0.12 * kappa**2 + term * np.log(tau / 0.25)
                grid_iv[i, j] = iv + np.random.normal(0, 0.001)
        surfaces.append(np.clip(grid_iv, 0.01, 5.0))
    return np.stack(surfaces, axis=0)

def create_sequences(data, lookback, horizon):
    T, M, N = data.shape
    num_samples = T - lookback - horizon + 1
    X = np.zeros((num_samples, lookback, 1, M, N), dtype=np.float32)
    y = np.zeros((num_samples, horizon, M, N), dtype=np.float32)
    for i in range(num_samples):
        X[i, :, 0, :, :] = data[i : i + lookback]
        y[i, :, :, :] = data[i + lookback : i + lookback + horizon]
    return X, y

class SurfaceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Training Script & Evaluation

In [19]:
# 7. Train Model Runner

def get_region_masks(kappas, taus):
    grid_k, grid_t = np.meshgrid(kappas, taus)
    return {
        "Overall": np.ones_like(grid_k, dtype=bool),
        "ATM": np.abs(grid_k) < 0.05,
        "OTM Puts": grid_k <= -0.10,
        "OTM Calls": grid_k >= 0.10,
        "Deep Wings": np.abs(grid_k) >= 0.20,
        "Short-dated": grid_t < 1/12,
        "Long-dated": grid_t > 6/12
    }

def calculate_metrics(y_pred, y_true):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mae = np.mean(np.abs(y_pred - y_true))
    mape = np.mean(np.abs(y_pred - y_true) / np.clip(y_true, 1e-5, None)) * 100.0
    tv_strike = np.mean(np.abs(y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]))
    tv_expiry = np.mean(np.abs(y_pred[:, :, :, 1:] - y_pred[:, :, :, :-1]))
    w_pred = (y_pred ** 2) * GRID_TAUS.reshape(1, 1, 1, -1)
    av_violations = np.sum(w_pred[:, :, :, :-1] > w_pred[:, :, :, 1:])
    av_ratio = av_violations / (y_pred.shape[0] * y_pred.shape[1] * y_pred.shape[2] * (y_pred.shape[3] - 1))
    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "TV": tv_strike + tv_expiry,
        "AV": av_ratio
    }

def train_model(model_name, X_train, y_train, X_val, y_val, lookback, horizon, epochs=30, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} on {device}...")
    
    if model_name == "convlstm":
        model = ConvLSTM(in_channels=1, hidden_dims=[32, 64, 64], num_layers=3, horizon=horizon)
    elif model_name == "lstm":
        model = StackedLSTM(grid_size=(7,7), hidden_dim=256, num_layers=3, horizon=horizon)
    elif model_name == "transformer":
        model = TransformerEncoderModel(grid_size=(7,7), horizon=horizon)
        
    model = model.to(device)
    criterion = SmoothnessRegularizedLoss(lambda_strike=0.01, lambda_expiry=0.01)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_loader = DataLoader(SurfaceDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SurfaceDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    
    best_val_loss = float("inf")
    best_path = f"{model_name}_best.pt"
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_accum = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            y_p = model(X_b)
            loss, _ = criterion(y_p, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item() * X_b.size(0)
            
        train_loss = train_loss_accum / len(train_loader.dataset)
        scheduler.step()
        
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                y_p = model(X_b)
                loss, _ = criterion(y_p, y_b)
                val_loss_accum += loss.item() * X_b.size(0)
        val_loss = val_loss_accum / len(val_loader.dataset)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model, best_path)
            
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch:02d} | Train Loss={train_loss:.6f} | Val Loss={val_loss:.6f}")
            
    print(f"Finished training {model_name}. Champion model loaded.")
    return torch.load(best_path, weights_only=False)

## Run Benchmarking Experiment

In [20]:
# 8. Run comparative evaluation

lookback = 20
horizon = 5
epochs = 20

# Generate datasets
dataset = generate_synthetic_dataset(num_days=300)
T = len(dataset)
train_end = int(T * 0.6)
val_end = int(T * 0.8)

train_data = dataset[:train_end]
val_data = dataset[train_end:val_end]
test_data = dataset[val_end:]

X_train, y_train = create_sequences(train_data, lookback, horizon)
X_val, y_val = create_sequences(val_data, lookback, horizon)
X_test, y_test = create_sequences(test_data, lookback, horizon)

masks = get_region_masks(GRID_KAPPAS, GRID_TAUS)
results = {}

# 1. Random Walk
rw = NaiveRandomWalk(horizon=horizon)
rw_pred = rw.predict(X_test)
results["Random Walk"] = calculate_metrics(rw_pred, y_test)

# 2. Historical Mean
hm = HistoricalMean(horizon=horizon)
hm_pred = hm.predict(X_test)
results["Historical Mean"] = calculate_metrics(hm_pred, y_test)

# 3. Exponential Smoothing
es = ExponentialSmoothing(horizon=horizon)
es_pred = es.predict(X_test)
results["Exp. Smoothing"] = calculate_metrics(es_pred, y_test)

# 4. HAR-RV
har = HARRVModel(horizon=horizon)
har.fit(X_train, y_train)
har_pred = har.predict(X_test)
results["HAR-RV"] = calculate_metrics(har_pred, y_test)

# 5. Stacked LSTM
lstm_model = train_model("lstm", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
lstm_model.eval()
with torch.no_grad():
    device = next(lstm_model.parameters()).device
    lstm_pred = lstm_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["LSTM"] = calculate_metrics(lstm_pred, y_test)

# 6. ConvLSTM (Smooth)
conv_model = train_model("convlstm", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
conv_model.eval()
with torch.no_grad():
    device = next(conv_model.parameters()).device
    conv_pred = conv_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["ConvLSTM (Smooth)"] = calculate_metrics(conv_pred, y_test)

# Save champion weights for local API serving
torch.save(conv_model.to("cpu"), "checkpoint.pt")
print("--> Successfully exported 'checkpoint.pt' model to active directory. Download this file to place in models/!")

Training lstm on cuda...
Epoch 05 | Train Loss=0.003075 | Val Loss=0.003652
Epoch 10 | Train Loss=0.002114 | Val Loss=0.003220
Epoch 15 | Train Loss=0.001676 | Val Loss=0.002013
Epoch 20 | Train Loss=0.001491 | Val Loss=0.001768
Finished training lstm. Champion model loaded.
Training convlstm on cuda...
Epoch 05 | Train Loss=0.001101 | Val Loss=0.016374
Epoch 10 | Train Loss=0.000742 | Val Loss=0.009918
Epoch 15 | Train Loss=0.000660 | Val Loss=0.005233
Epoch 20 | Train Loss=0.000710 | Val Loss=0.001222
Finished training convlstm. Champion model loaded.
--> Successfully exported 'checkpoint.pt' model to active directory. Download this file to place in models/!


## Display Final Benchmark Table

In [22]:
# 9. Print results
print("\n" + "="*90)
print(f"{'Model':<20} | {'RMSE':<8} | {'MAE':<8} | {'TV (Roughness)':<15} | {'AV (Arbitrage Violations)':<25}")
print("-"*90)
for model_name, metrics in results.items():
    print(f"{model_name:<20} | {metrics['RMSE']:.5f} | {metrics['MAE']:.5f} | {metrics['TV']:.5f}         | {metrics['AV']:.4%}")
print("="*90)


Model                | RMSE     | MAE      | TV (Roughness)  | AV (Arbitrage Violations)
------------------------------------------------------------------------------------------
Random Walk          | 0.02685 | 0.02010 | 0.03487         | 0.3307%
Historical Mean      | 0.03108 | 0.02576 | 0.03800         | 0.0000%
Exp. Smoothing       | 0.02825 | 0.02196 | 0.03596         | 0.0000%
HAR-RV               | 0.02702 | 0.02038 | 0.03331         | 0.0529%
LSTM                 | 0.03490 | 0.02897 | 0.02744         | 0.0000%
ConvLSTM (Smooth)    | 0.03139 | 0.02436 | 0.02786         | 0.0397%
